In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [5]:
# ! pip install byaldi
# ! pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 104.4 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [21]:
# from sentence_transformers import MultiVectorEncoder
from byaldi import RAGMultiModalModel
from datasets import load_dataset
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import os

# Folder Paths

In [2]:
ROOT = "/content/drive/MyDrive/ColPali IR project"
PDF_PATH = ROOT + "/Data/Arxive PDFs"
IMG_PATH = ROOT + "/Data/Pages/Images"
TXT_PATH = ROOT + "/Data/Pages/Texts"
QRS_PATH = ROOT + "/Data/Queries/colpali_50_queries.csv"
ARXIVQA_IMG = ROOT + "/Data/ArxiveQA/Images"
DOCQA_IMG = ROOT + "/Data/DocQA/Images"

# ColQwen Inference on ArxiveQA and docQA

In [3]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device : {device}")

Device : cuda


# Dataset Loading

In [11]:
# Data loading

docvqa = load_dataset(
    "vidore/docvqa_test_subsampled", split="test"
)

arxivqa = load_dataset(
    "vidore/arxivqa_test_subsampled", split="test"
)

# Preview
print(docvqa)
print(arxivqa)

Dataset({
    features: ['questionId', 'query', 'question_types', 'image', 'docId', 'image_filename', 'page', 'answer', 'data_split', 'source'],
    num_rows: 500
})
Dataset({
    features: ['query', 'image', 'image_filename', 'options', 'answer', 'page', 'model', 'prompt', 'source'],
    num_rows: 500
})


In [11]:
# Saving the images of ArxivQA and docVQA for indexing and inference

arxivqa_img_path = Path(ARXIVQA_IMG)
docqa_img_path = Path(DOCQA_IMG)

for i, sample in enumerate(arxivqa):
    image = sample["image"]
    image.convert("RGB").save(arxivqa_img_path / f"arxivqa_page_{i+1:03d}.png")

for i, sample in enumerate(docvqa):
    image = sample["image"]
    image.convert("RGB").save(docqa_img_path / f"docvqa_page_{i+1:03d}.png")

# Queries dataset

In [82]:
arxivqa_queries = pd.DataFrame({"query": arxivqa["query"], "page_path": os.listdir(ARXIVQA_IMG)})
docvqa_queries = pd.DataFrame({"query": docvqa["query"], "page_path": os.listdir(DOCQA_IMG)})

# Preview
display(arxivqa_queries.head())
display(docvqa_queries.head())

,query,page_path
0,"Based on the graph, what is the impact of corr...",arxivqa_page_001.png
1,"Based on the progression from JUL10 to FEB11Q,...",arxivqa_page_002.png
2,What physical phenomenon could the pattern of ...,arxivqa_page_003.png
3,What is the approximate uv-distance where the ...,arxivqa_page_004.png
4,In which subfigures does the average arrival f...,arxivqa_page_005.png


,query,page_path
0,What is the dividend payout in 2012?,docvqa_page_001.png
1,What is the name of the person in the CC field ?,docvqa_page_002.png
2,What is the personnel costs in the 4th year?,docvqa_page_003.png
3,What is the table number?,docvqa_page_004.png
4,What is the Log-in No. ?,docvqa_page_005.png


# DocVQA and ArxivQA Indexing

In [39]:
# DocVQA
model_docvqa = RAGMultiModalModel.from_pretrained(
    "vidore/colqwen2-v1.0",
    device=device
)

# Indexing DocVQA
model_docvqa.index(
    input_path=DOCQA_IMG,
    index_name="docvqa_index",
    overwrite=True,
)

Verbosity is set to 1 (active). Pass verbose=0 to make quieter.


Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

overwrite is on. Deleting existing index docvqa_index to build a new one.
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_001.png
Added page 1 of document 0 to index.
Index exported to .byaldi/docvqa_index
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_002.png
Added page 1 of document 1 to index.
Index exported to .byaldi/docvqa_index
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_003.png
Added page 1 of document 2 to index.
Index exported to .byaldi/docvqa_index
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_004.png
Added page 1 of document 3 to index.
Index exported to .byaldi/docvqa_index
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_005.png
Added page 1 of document 4 to index.
Index exported to .byaldi/docvqa_index
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/DocQA/Ima

{0: '/content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_001.png',
 1: '/content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_002.png',
 2: '/content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_003.png',
 3: '/content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_004.png',
 4: '/content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_005.png',
 5: '/content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_006.png',
 6: '/content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_007.png',
 7: '/content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_008.png',
 8: '/content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_009.png',
 9: '/content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_010.png',
 10: '/content/drive/MyDrive/ColPali IR project/Data/DocQA/Images/docvqa_page_011.png',
 11: '/content/drive/MyDrive/ColPali IR pr

In [31]:
# ArxivQA
model_arxivqa = RAGMultiModalModel.from_pretrained(
    "vidore/colqwen2-v1.0"
)

# Indexing ArxivQA
model_arxivqa.index(
    input_path=ARXIVQA_IMG,
    index_name="arxivqa_index",
    overwrite=True
)

Verbosity is set to 1 (active). Pass verbose=0 to make quieter.


Loading weights:   0%|          | 0/731 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/394 [00:00<?, ?it/s]

Indexing file: /content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_001.png
Added page 1 of document 0 to index.
Index exported to .byaldi/arxivqa_index
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_002.png
Added page 1 of document 1 to index.
Index exported to .byaldi/arxivqa_index
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_003.png
Added page 1 of document 2 to index.
Index exported to .byaldi/arxivqa_index
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_004.png
Added page 1 of document 3 to index.
Index exported to .byaldi/arxivqa_index
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_005.png
Added page 1 of document 4 to index.
Index exported to .byaldi/arxivqa_index
Indexing file: /content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_006.png
Added page 1 of docum

{0: '/content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_001.png',
 1: '/content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_002.png',
 2: '/content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_003.png',
 3: '/content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_004.png',
 4: '/content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_005.png',
 5: '/content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_006.png',
 6: '/content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_007.png',
 7: '/content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_008.png',
 8: '/content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_009.png',
 9: '/content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_010.png',
 10: '/content/drive/MyDrive/ColPali IR project/Data/ArxiveQA/Images/arxivqa_page_011.png'

# Retrieval

In [56]:
# Retrieval function

def retrieval(df, model, k=5):

    all_retrieved = []
    all_scores = []

    for idx, row in df.iterrows():

        query_text = row["query"]
        results = model.search(query_text, k=k)

        retrieved = [r["doc_id"] for r in results]
        scores = [r["score"] for r in results]

        all_retrieved.append(retrieved)
        all_scores.append(scores)

    return np.array(all_retrieved), all_scores

In [57]:
# top-5 retrieved documents and their scores
all_retrieved_docvqa, all_scores_docvqa = retrieval(docvqa_queries, model_docvqa, k=5)

In [58]:
# top-5 retrieved documents and their scores
all_retrieved_arxivqa, all_scores_docvqa = retrieval(arxivqa_queries, model_arxivqa, k=5)

# Evaluation

In [65]:
# Compute Discounted Cumulative Gain (DCG) and normalized DCG (nDCG) at K

def dcg_at_k(rels, k=5):

    rels = np.array(rels[:k], dtype=float)
    return float(np.sum((2**rels - 1) / np.log2(np.arange(1, len(rels)+1) + 1)))

# Compute nDCG@K assuming a single relevant ground-truth document target
def ndcg_at_k(retrieved, relevant, k=5):

    rels = [1 if d in relevant else 0 for d in retrieved[:k]]
    dcg = dcg_at_k(rels, k)

    ideal_rels = [1] + [0] * (k - 1)
    idcg = dcg_at_k(ideal_rels, k)

    return dcg / idcg if idcg > 0 else 0.0

In [66]:
# Compute nDCG@5
relevant = np.arange(len(arxivqa_queries))
ndcg5 = np.mean([ndcg_at_k(retrieved, relevant, k=5) for retrieved, relevant in zip(all_retrieved_docvqa, [[r] for r in relevant])])
print(f"DocVQA nDCG@5 : {ndcg5}")

DocVQA nDCG@5 : 0.5486104402650461


In [67]:
# Compute nDCG@5
ndcg5_arxivqa = np.mean([ndcg_at_k(retrieved, relevant, k=5) for retrieved, relevant in zip(all_retrieved_arxivqa, [[r] for r in relevant])])
print(f"ArxivQA nDCG@5 : {ndcg5_arxivqa}")

ArxivQA nDCG@5 : 0.8763568400622681
